In [0]:
%pip install langdetect

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
from langdetect import detect, detect_langs

In [0]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, lower, trim, regexp_replace,
    length, sha2, concat_ws, when, to_timestamp, monotonically_increasing_id
)

from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

from langdetect import detect

In [0]:
# Chemins possibles selon l'emplacement du dossier data dans Databricks

possible_tweets_paths = [
    "/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv",
    "data/fifa_world_cup_2022_tweets.csv",
    "../data/fifa_world_cup_2022_tweets.csv",
    "/FileStore/tables/fifa_world_cup_2022_tweets.csv"
]

possible_events_paths = [
    "data/match_events.csv",
    "../data/match_events.csv",
    "/FileStore/tables/match_events.csv"
]

print("Chemins possibles tweets :")
for p in possible_tweets_paths:
    print("-", p)

Chemins possibles tweets :
- /Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv
- data/fifa_world_cup_2022_tweets.csv
- ../data/fifa_world_cup_2022_tweets.csv
- /FileStore/tables/fifa_world_cup_2022_tweets.csv


In [0]:
def read_csv_first_available(paths):
    """
    Essaie plusieurs chemins et lit le premier fichier CSV disponible.
    """
    last_error = None
    
    for path in paths:
        try:
            df = (
                spark.read
                .option("header", True)
                .option("inferSchema", True)
                .option("multiLine", True)
                .option("escape", '"')
                .csv(path)
            )
            
            # On force une action pour vérifier que le fichier est lisible
            count_rows = df.count()
            print(f"Fichier chargé avec succès : {path}")
            print(f"Nombre de lignes : {count_rows}")
            return df, path
        
        except Exception as e:
            last_error = e
            print(f"Chemin non valide : {path}")
    
    raise Exception(f"Aucun fichier trouvé. Dernière erreur : {last_error}")

In [0]:
import os
 
print("Dossier actuel du notebook :")

print(os.getcwd())
 
print("\nFichiers/dossiers ici :")

print(os.listdir("."))
 

Dossier actuel du notebook :
/Workspace/Users/bentissesalma18@gmail.com/TP (1)

Fichiers/dossiers ici :
['New Notebook 2026-07-02 12:44:22.ipynb', 'match_events.csv', 'fifa_world_cup_2022_tweets.csv']


In [0]:
tweets_path_used = "file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv"
 
df_raw = (

    spark.read

    .option("header", True)

    .option("inferSchema", True)

    .option("multiLine", True)

    .option("escape", '"')

    .csv(tweets_path_used)

)
 
print("Fichier utilisé :", tweets_path_used)
 
display(df_raw.limit(10))
 

Fichier utilisé : file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv


_c0,Date Created,Number of Likes,Source of Tweet,Tweet,Sentiment
0,2022-11-20T23:59:21.000Z,4,Twitter Web App,What are we drinking today @TucanTribe @MadBears_ @lkinc_algo @al_goanna #WorldCup2022 https://t.co/Oga3TzvG5h,neutral
1,2022-11-20T23:59:01.000Z,3,Twitter for iPhone,Amazing @CanadaSoccerEN #WorldCup2022 launch video. Shows how much the face of Canada and our men’s national team have changed since our last World Cup entry in 1986. Can’t wait to see these boys in action! This is Canada: FIFA World Cup Opening Video https://t.co/7g73vvwtg8,positive
2,2022-11-20T23:58:41.000Z,1,Twitter for iPhone,Worth reading while watching #WorldCup2022 https://t.co/1SQrNa2dYU,positive
3,2022-11-20T23:58:33.000Z,1,Twitter Web App,Golden Maknae shinning bright https://t.co/4AyZbzGTX4 #JeonJungkook #Jungkook #전정국 #정국 #JK #GoldenMaknae #bunny #Kookie #Jungshook #BTS #방탄소년단 #WorldCup2022 #FIFAKOOK @BTS_twt,positive
4,2022-11-20T23:58:28.000Z,0,Twitter for Android,"If the BBC cares so much about human rights, homosexual rights, and women rights then why not say these before the opening ceremony?? Why are they saying these during the opening ceremony?? Why did the BBC censor the #WorldCup2022 opening ceremony?? https://t.co/f72P03ZN2k",negative
5,2022-11-20T23:57:32.000Z,0,Twitter for Android,"And like, will the mexican fans be able to scream ""PUTO"" now? Or is that too homophobic for qatar? @FIFAWorldCup #WorldCup2022",negative
6,2022-11-20T23:57:06.000Z,0,Twitter for Android,Look like a only me and the Jamaican football team naw follow up worldcup #WorldCup2022,neutral
7,2022-11-20T23:57:05.000Z,0,Twitter for Android,Really? Football on a Monday morning at 9 and 12 and then 3? I need to pinch myself. Is it really happening? #WorldCup2022,negative
8,2022-11-20T23:56:10.000Z,1,Twitter for iPhone,"As the World Cup starts in Qatar, it’s Black Awareness Day in Brazil✊🏽. Despite the atrocities linked to this year’s host and to Fifa, soccer is so fundamental for lower classes, mostly Black and Brown, as Vini Jr identifies #WorldCup2022 #diadaconsciencianegra #CopaDoMundo2022 https://t.co/8mI9UqNWX7",positive
9,2022-11-20T23:56:08.000Z,0,Twitter for iPhone,#WorldCup2022 @ITVSport & @LFSYSTEMMUSIC go together so well ‘Hungry For Your Love’🫶 ⚽️👌,positive


In [0]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, lower, trim, regexp_replace,
    length, sha2, concat_ws, when, to_timestamp, monotonically_increasing_id,
    count, desc
)

from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

from langdetect import detect

In [0]:
tweets_path = "file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv"
events_path = "file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/match_events.csv"

print("Chemin tweets :", tweets_path)
print("Chemin événements :", events_path)

Chemin tweets : file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv
Chemin événements : file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/match_events.csv


In [0]:
df_tweets_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(tweets_path)
)

display(df_tweets_raw.limit(10))

_c0,Date Created,Number of Likes,Source of Tweet,Tweet,Sentiment
0,2022-11-20T23:59:21.000Z,4,Twitter Web App,What are we drinking today @TucanTribe @MadBears_ @lkinc_algo @al_goanna #WorldCup2022 https://t.co/Oga3TzvG5h,neutral
1,2022-11-20T23:59:01.000Z,3,Twitter for iPhone,Amazing @CanadaSoccerEN #WorldCup2022 launch video. Shows how much the face of Canada and our men’s national team have changed since our last World Cup entry in 1986. Can’t wait to see these boys in action! This is Canada: FIFA World Cup Opening Video https://t.co/7g73vvwtg8,positive
2,2022-11-20T23:58:41.000Z,1,Twitter for iPhone,Worth reading while watching #WorldCup2022 https://t.co/1SQrNa2dYU,positive
3,2022-11-20T23:58:33.000Z,1,Twitter Web App,Golden Maknae shinning bright https://t.co/4AyZbzGTX4 #JeonJungkook #Jungkook #전정국 #정국 #JK #GoldenMaknae #bunny #Kookie #Jungshook #BTS #방탄소년단 #WorldCup2022 #FIFAKOOK @BTS_twt,positive
4,2022-11-20T23:58:28.000Z,0,Twitter for Android,"If the BBC cares so much about human rights, homosexual rights, and women rights then why not say these before the opening ceremony?? Why are they saying these during the opening ceremony?? Why did the BBC censor the #WorldCup2022 opening ceremony?? https://t.co/f72P03ZN2k",negative
5,2022-11-20T23:57:32.000Z,0,Twitter for Android,"And like, will the mexican fans be able to scream ""PUTO"" now? Or is that too homophobic for qatar? @FIFAWorldCup #WorldCup2022",negative
6,2022-11-20T23:57:06.000Z,0,Twitter for Android,Look like a only me and the Jamaican football team naw follow up worldcup #WorldCup2022,neutral
7,2022-11-20T23:57:05.000Z,0,Twitter for Android,Really? Football on a Monday morning at 9 and 12 and then 3? I need to pinch myself. Is it really happening? #WorldCup2022,negative
8,2022-11-20T23:56:10.000Z,1,Twitter for iPhone,"As the World Cup starts in Qatar, it’s Black Awareness Day in Brazil✊🏽. Despite the atrocities linked to this year’s host and to Fifa, soccer is so fundamental for lower classes, mostly Black and Brown, as Vini Jr identifies #WorldCup2022 #diadaconsciencianegra #CopaDoMundo2022 https://t.co/8mI9UqNWX7",positive
9,2022-11-20T23:56:08.000Z,0,Twitter for iPhone,#WorldCup2022 @ITVSport & @LFSYSTEMMUSIC go together so well ‘Hungry For Your Love’🫶 ⚽️👌,positive


In [0]:
df_events_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(events_path)
)

display(df_events_raw.limit(10))

In [0]:
print("===== Schéma tweets =====")
df_tweets_raw.printSchema()

print("===== Schéma événements =====")
df_events_raw.printSchema()

===== Schéma tweets =====
root
 |-- _c0: integer (nullable = true)
 |-- Date Created: timestamp (nullable = true)
 |-- Number of Likes: integer (nullable = true)
 |-- Source of Tweet: string (nullable = true)
 |-- Tweet: string (nullable = true)
 |-- Sentiment: string (nullable = true)

===== Schéma événements =====
root



In [0]:
print("Colonnes tweets :")
for c in df_tweets_raw.columns:
    print("-", c)

print("\nColonnes événements :")
for c in df_events_raw.columns:
    print("-", c)

Colonnes tweets :
- _c0
- Date Created
- Number of Likes
- Source of Tweet
- Tweet
- Sentiment

Colonnes événements :


In [0]:
def find_column(df, candidates):
    """
    Trouve une colonne dans un DataFrame à partir d'une liste de noms possibles.
    La comparaison ignore les espaces, underscores et majuscules.
    """
    normalized_columns = {
        c.lower().replace(" ", "").replace("_", ""): c
        for c in df.columns
    }
    
    for candidate in candidates:
        key = candidate.lower().replace(" ", "").replace("_", "")
        if key in normalized_columns:
            return normalized_columns[key]
    
    return None

In [0]:
tweet_id_col = find_column(df_tweets_raw, [

    "_c0", "Tweet Id", "TweetID", "tweet_id", "id", "post_id"

])
 
tweet_text_col = find_column(df_tweets_raw, [

    "Tweet", "Text", "tweet", "content", "message", "text"

])
 
tweet_date_col = find_column(df_tweets_raw, [

    "Date Created", "Datetime", "Date", "created_at", "timestamp", "Created At"

])
 
tweet_source_col = find_column(df_tweets_raw, [

    "Source of Tweet", "source", "Source"

])
 
tweet_likes_col = find_column(df_tweets_raw, [

    "Number of Likes", "likes", "like_count"

])
 
existing_sentiment_col = find_column(df_tweets_raw, [

    "Sentiment", "sentiment"

])
 
print("Colonnes détectées pour tweets :")

print("tweet_id_col          =", tweet_id_col)

print("tweet_text_col        =", tweet_text_col)

print("tweet_date_col        =", tweet_date_col)

print("tweet_source_col      =", tweet_source_col)

print("tweet_likes_col       =", tweet_likes_col)

print("existing_sentiment_col=", existing_sentiment_col)
 

Colonnes détectées pour tweets :
tweet_id_col          = _c0
tweet_text_col        = Tweet
tweet_date_col        = Date Created
tweet_source_col      = Source of Tweet
tweet_likes_col       = Number of Likes
existing_sentiment_col= Sentiment


In [0]:
tweet_id_col = find_column(df_tweets_raw, [
    "_c0", "Tweet Id", "TweetID", "tweet_id", "id", "post_id"
])

tweet_text_col = find_column(df_tweets_raw, [
    "Tweet", "Text", "tweet", "content", "message", "text"
])

tweet_date_col = find_column(df_tweets_raw, [
    "Date Created", "Datetime", "Date", "created_at", "timestamp", "Created At"
])

tweet_source_col = find_column(df_tweets_raw, [
    "Source of Tweet", "source", "Source"
])

tweet_likes_col = find_column(df_tweets_raw, [
    "Number of Likes", "likes", "like_count"
])

existing_sentiment_col = find_column(df_tweets_raw, [
    "Sentiment", "sentiment"
])

print("Colonnes détectées pour tweets :")
print("tweet_id_col           =", tweet_id_col)
print("tweet_text_col         =", tweet_text_col)
print("tweet_date_col         =", tweet_date_col)
print("tweet_source_col       =", tweet_source_col)
print("tweet_likes_col        =", tweet_likes_col)
print("existing_sentiment_col =", existing_sentiment_col)

Colonnes détectées pour tweets :
tweet_id_col           = _c0
tweet_text_col         = Tweet
tweet_date_col         = Date Created
tweet_source_col       = Source of Tweet
tweet_likes_col        = Number of Likes
existing_sentiment_col = Sentiment


In [0]:
if tweet_text_col is None:
    raise Exception("Impossible de trouver la colonne du texte dans le fichier tweets.")

if tweet_date_col is None:
    print("Attention : colonne date non trouvée dans le fichier tweets.")

if tweet_id_col is None:
    print("Attention : colonne tweet_id non trouvée. Un identifiant technique sera généré.")

if tweet_source_col is None:
    print("Attention : colonne source non trouvée. La source sera mise à 'twitter'.")

if tweet_likes_col is None:
    print("Attention : colonne likes non trouvée. Elle sera mise à null.")

if existing_sentiment_col is None:
    print("Attention : colonne sentiment source non trouvée. Elle sera mise à null.")

In [0]:
if tweet_id_col is not None:
    tweet_id_expr = col(tweet_id_col).cast("string")
else:
    tweet_id_expr = monotonically_increasing_id().cast("string")

if tweet_date_col is not None:
    tweet_date_expr = col(tweet_date_col).cast("string")
else:
    tweet_date_expr = lit(None).cast("string")

if tweet_source_col is not None:
    tweet_source_expr = col(tweet_source_col).cast("string")
else:
    tweet_source_expr = lit("twitter")

if tweet_likes_col is not None:
    tweet_likes_expr = col(tweet_likes_col).cast("int")
else:
    tweet_likes_expr = lit(None).cast("int")

if existing_sentiment_col is not None:
    existing_sentiment_expr = col(existing_sentiment_col).cast("string")
else:
    existing_sentiment_expr = lit(None).cast("string")

df_tweets_standard = df_tweets_raw.select(
    tweet_id_expr.alias("post_id"),
    tweet_source_expr.alias("source"),
    tweet_date_expr.alias("created_at_raw"),
    lit("world_cup_2022").alias("match_name"),
    col(tweet_text_col).cast("string").alias("text_raw"),
    tweet_likes_expr.alias("number_of_likes"),
    existing_sentiment_expr.alias("sentiment_source"),
    lit(tweets_path).alias("bronze_source_file"),
    current_timestamp().alias("ingestion_ts")
)

display(df_tweets_standard.limit(10))

post_id,source,created_at_raw,match_name,text_raw,number_of_likes,sentiment_source,bronze_source_file,ingestion_ts
0,Twitter Web App,2022-11-20 23:59:21,world_cup_2022,What are we drinking today @TucanTribe @MadBears_ @lkinc_algo @al_goanna #WorldCup2022 https://t.co/Oga3TzvG5h,4,neutral,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z
1,Twitter for iPhone,2022-11-20 23:59:01,world_cup_2022,Amazing @CanadaSoccerEN #WorldCup2022 launch video. Shows how much the face of Canada and our men’s national team have changed since our last World Cup entry in 1986. Can’t wait to see these boys in action! This is Canada: FIFA World Cup Opening Video https://t.co/7g73vvwtg8,3,positive,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z
2,Twitter for iPhone,2022-11-20 23:58:41,world_cup_2022,Worth reading while watching #WorldCup2022 https://t.co/1SQrNa2dYU,1,positive,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z
3,Twitter Web App,2022-11-20 23:58:33,world_cup_2022,Golden Maknae shinning bright https://t.co/4AyZbzGTX4 #JeonJungkook #Jungkook #전정국 #정국 #JK #GoldenMaknae #bunny #Kookie #Jungshook #BTS #방탄소년단 #WorldCup2022 #FIFAKOOK @BTS_twt,1,positive,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z
4,Twitter for Android,2022-11-20 23:58:28,world_cup_2022,"If the BBC cares so much about human rights, homosexual rights, and women rights then why not say these before the opening ceremony?? Why are they saying these during the opening ceremony?? Why did the BBC censor the #WorldCup2022 opening ceremony?? https://t.co/f72P03ZN2k",0,negative,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z
5,Twitter for Android,2022-11-20 23:57:32,world_cup_2022,"And like, will the mexican fans be able to scream ""PUTO"" now? Or is that too homophobic for qatar? @FIFAWorldCup #WorldCup2022",0,negative,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z
6,Twitter for Android,2022-11-20 23:57:06,world_cup_2022,Look like a only me and the Jamaican football team naw follow up worldcup #WorldCup2022,0,neutral,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z
7,Twitter for Android,2022-11-20 23:57:05,world_cup_2022,Really? Football on a Monday morning at 9 and 12 and then 3? I need to pinch myself. Is it really happening? #WorldCup2022,0,negative,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z
8,Twitter for iPhone,2022-11-20 23:56:10,world_cup_2022,"As the World Cup starts in Qatar, it’s Black Awareness Day in Brazil✊🏽. Despite the atrocities linked to this year’s host and to Fifa, soccer is so fundamental for lower classes, mostly Black and Brown, as Vini Jr identifies #WorldCup2022 #diadaconsciencianegra #CopaDoMundo2022 https://t.co/8mI9UqNWX7",1,positive,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z
9,Twitter for iPhone,2022-11-20 23:56:08,world_cup_2022,#WorldCup2022 @ITVSport & @LFSYSTEMMUSIC go together so well ‘Hungry For Your Love’🫶 ⚽️👌,0,positive,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:51:50.055Z


In [0]:
df_tweets_standard = df_tweets_standard.withColumn(
    "created_at",
    to_timestamp(col("created_at_raw"))
)

display(df_tweets_standard.select("created_at_raw", "created_at").limit(10))

created_at_raw,created_at
2022-11-20 23:59:21,2022-11-20T23:59:21.000Z
2022-11-20 23:59:01,2022-11-20T23:59:01.000Z
2022-11-20 23:58:41,2022-11-20T23:58:41.000Z
2022-11-20 23:58:33,2022-11-20T23:58:33.000Z
2022-11-20 23:58:28,2022-11-20T23:58:28.000Z
2022-11-20 23:57:32,2022-11-20T23:57:32.000Z
2022-11-20 23:57:06,2022-11-20T23:57:06.000Z
2022-11-20 23:57:05,2022-11-20T23:57:05.000Z
2022-11-20 23:56:10,2022-11-20T23:56:10.000Z
2022-11-20 23:56:08,2022-11-20T23:56:08.000Z


In [0]:
df_tweets_clean = df_tweets_standard.withColumn(
    "text_clean",
    lower(trim(col("text_raw")))
)

# Supprimer les liens
df_tweets_clean = df_tweets_clean.withColumn(
    "text_clean",
    regexp_replace(col("text_clean"), r"http\S+|www\S+", "")
)

# Supprimer les mentions @user
df_tweets_clean = df_tweets_clean.withColumn(
    "text_clean",
    regexp_replace(col("text_clean"), r"@\w+", "")
)

# Supprimer RT au début des retweets
df_tweets_clean = df_tweets_clean.withColumn(
    "text_clean",
    regexp_replace(col("text_clean"), r"^rt\s+", "")
)

# Supprimer retours ligne / tabulations
df_tweets_clean = df_tweets_clean.withColumn(
    "text_clean",
    regexp_replace(col("text_clean"), r"[\n\r\t]", " ")
)

# Réduire les espaces multiples
df_tweets_clean = df_tweets_clean.withColumn(
    "text_clean",
    regexp_replace(col("text_clean"), r"\s+", " ")
)

# Trim final
df_tweets_clean = df_tweets_clean.withColumn(
    "text_clean",
    trim(col("text_clean"))
)

# Longueur texte
df_tweets_clean = df_tweets_clean.withColumn(
    "text_length",
    length(col("text_clean"))
)

display(df_tweets_clean.select("text_raw", "text_clean", "text_length").limit(20))

text_raw,text_clean,text_length
What are we drinking today @TucanTribe @MadBears_ @lkinc_algo @al_goanna #WorldCup2022 https://t.co/Oga3TzvG5h,what are we drinking today #worldcup2022,40
Amazing @CanadaSoccerEN #WorldCup2022 launch video. Shows how much the face of Canada and our men’s national team have changed since our last World Cup entry in 1986. Can’t wait to see these boys in action! This is Canada: FIFA World Cup Opening Video https://t.co/7g73vvwtg8,amazing #worldcup2022 launch video. shows how much the face of canada and our men’s national team have changed since our last world cup entry in 1986. can’t wait to see these boys in action! this is canada: fifa world cup opening video,235
Worth reading while watching #WorldCup2022 https://t.co/1SQrNa2dYU,worth reading while watching #worldcup2022,42
Golden Maknae shinning bright https://t.co/4AyZbzGTX4 #JeonJungkook #Jungkook #전정국 #정국 #JK #GoldenMaknae #bunny #Kookie #Jungshook #BTS #방탄소년단 #WorldCup2022 #FIFAKOOK @BTS_twt,golden maknae shinning bright #jeonjungkook #jungkook #전정국 #정국 #jk #goldenmaknae #bunny #kookie #jungshook #bts #방탄소년단 #worldcup2022 #fifakook,142
"If the BBC cares so much about human rights, homosexual rights, and women rights then why not say these before the opening ceremony?? Why are they saying these during the opening ceremony?? Why did the BBC censor the #WorldCup2022 opening ceremony?? https://t.co/f72P03ZN2k","if the bbc cares so much about human rights, homosexual rights, and women rights then why not say these before the opening ceremony?? why are they saying these during the opening ceremony?? why did the bbc censor the #worldcup2022 opening ceremony??",249
"And like, will the mexican fans be able to scream ""PUTO"" now? Or is that too homophobic for qatar? @FIFAWorldCup #WorldCup2022","and like, will the mexican fans be able to scream ""puto"" now? or is that too homophobic for qatar? #worldcup2022",112
Look like a only me and the Jamaican football team naw follow up worldcup #WorldCup2022,look like a only me and the jamaican football team naw follow up worldcup #worldcup2022,87
Really? Football on a Monday morning at 9 and 12 and then 3? I need to pinch myself. Is it really happening? #WorldCup2022,really? football on a monday morning at 9 and 12 and then 3? i need to pinch myself. is it really happening? #worldcup2022,122
"As the World Cup starts in Qatar, it’s Black Awareness Day in Brazil✊🏽. Despite the atrocities linked to this year’s host and to Fifa, soccer is so fundamental for lower classes, mostly Black and Brown, as Vini Jr identifies #WorldCup2022 #diadaconsciencianegra #CopaDoMundo2022 https://t.co/8mI9UqNWX7","as the world cup starts in qatar, it’s black awareness day in brazil✊🏽. despite the atrocities linked to this year’s host and to fifa, soccer is so fundamental for lower classes, mostly black and brown, as vini jr identifies #worldcup2022 #diadaconsciencianegra #copadomundo2022",278
#WorldCup2022 @ITVSport & @LFSYSTEMMUSIC go together so well ‘Hungry For Your Love’🫶 ⚽️👌,#worldcup2022 & go together so well ‘hungry for your love’🫶 ⚽️👌,67


In [0]:
df_tweets_clean = df_tweets_clean.withColumn(
    "has_url",
    when(col("text_raw").rlike(r"http\S+|www\S+"), True).otherwise(False)
)

df_tweets_clean = df_tweets_clean.withColumn(
    "has_hashtag",
    when(col("text_raw").rlike(r"#\w+"), True).otherwise(False)
)

display(df_tweets_clean.select("text_raw", "has_url", "has_hashtag").limit(20))

text_raw,has_url,has_hashtag
What are we drinking today @TucanTribe @MadBears_ @lkinc_algo @al_goanna #WorldCup2022 https://t.co/Oga3TzvG5h,true,true
Amazing @CanadaSoccerEN #WorldCup2022 launch video. Shows how much the face of Canada and our men’s national team have changed since our last World Cup entry in 1986. Can’t wait to see these boys in action! This is Canada: FIFA World Cup Opening Video https://t.co/7g73vvwtg8,true,true
Worth reading while watching #WorldCup2022 https://t.co/1SQrNa2dYU,true,true
Golden Maknae shinning bright https://t.co/4AyZbzGTX4 #JeonJungkook #Jungkook #전정국 #정국 #JK #GoldenMaknae #bunny #Kookie #Jungshook #BTS #방탄소년단 #WorldCup2022 #FIFAKOOK @BTS_twt,true,true
"If the BBC cares so much about human rights, homosexual rights, and women rights then why not say these before the opening ceremony?? Why are they saying these during the opening ceremony?? Why did the BBC censor the #WorldCup2022 opening ceremony?? https://t.co/f72P03ZN2k",true,true
"And like, will the mexican fans be able to scream ""PUTO"" now? Or is that too homophobic for qatar? @FIFAWorldCup #WorldCup2022",false,true
Look like a only me and the Jamaican football team naw follow up worldcup #WorldCup2022,false,true
Really? Football on a Monday morning at 9 and 12 and then 3? I need to pinch myself. Is it really happening? #WorldCup2022,false,true
"As the World Cup starts in Qatar, it’s Black Awareness Day in Brazil✊🏽. Despite the atrocities linked to this year’s host and to Fifa, soccer is so fundamental for lower classes, mostly Black and Brown, as Vini Jr identifies #WorldCup2022 #diadaconsciencianegra #CopaDoMundo2022 https://t.co/8mI9UqNWX7",true,true
#WorldCup2022 @ITVSport & @LFSYSTEMMUSIC go together so well ‘Hungry For Your Love’🫶 ⚽️👌,false,true


In [0]:
df_tweets_key = df_tweets_clean.withColumn(
    "duplicate_key",
    sha2(
        concat_ws(
            "||",
            col("source"),
            col("text_clean"),
            col("created_at_raw")
        ),
        256
    )
)

display(df_tweets_key.select("post_id", "text_clean", "duplicate_key").limit(10))

post_id,text_clean,duplicate_key
0,what are we drinking today #worldcup2022,4f24da4662726ac2406d283668ca74eb90217465e2b84b2265d449fb2514b305
1,amazing #worldcup2022 launch video. shows how much the face of canada and our men’s national team have changed since our last world cup entry in 1986. can’t wait to see these boys in action! this is canada: fifa world cup opening video,94e81f5129f750a27369268b6b4be41c0b69f3e36b5067ba09c48853724a668e
2,worth reading while watching #worldcup2022,e84e819df71acf656bf5fd10ad1f3ed741c48575120e2f9434325a7a69d8fc3b
3,golden maknae shinning bright #jeonjungkook #jungkook #전정국 #정국 #jk #goldenmaknae #bunny #kookie #jungshook #bts #방탄소년단 #worldcup2022 #fifakook,9cbfc76c8e6765d0ff6584022adeedf52313fc43e30e3b86849b1c81d65cd61e
4,"if the bbc cares so much about human rights, homosexual rights, and women rights then why not say these before the opening ceremony?? why are they saying these during the opening ceremony?? why did the bbc censor the #worldcup2022 opening ceremony??",0ccec0311cff639f094113ec92e7aef5486acdd057068f9c4cc02073509e2d8f
5,"and like, will the mexican fans be able to scream ""puto"" now? or is that too homophobic for qatar? #worldcup2022",6a3a4401879da603fd81a564cacce9de2896ced26c9f32fb0c06004033709d2c
6,look like a only me and the jamaican football team naw follow up worldcup #worldcup2022,27d40dfa2fca41f2f9e868f2a4218c57d03b65a86191a43142dc340796359b1e
7,really? football on a monday morning at 9 and 12 and then 3? i need to pinch myself. is it really happening? #worldcup2022,13b498bd587792f22a946c77d4dbfad0121eed46c105433563e5d1cd0f04af4b
8,"as the world cup starts in qatar, it’s black awareness day in brazil✊🏽. despite the atrocities linked to this year’s host and to fifa, soccer is so fundamental for lower classes, mostly black and brown, as vini jr identifies #worldcup2022 #diadaconsciencianegra #copadomundo2022",81caa1c74bbdf8907c5df5d2e0317196cf214697ba677e6c46579ffd3953a5bd
9,#worldcup2022 & go together so well ‘hungry for your love’🫶 ⚽️👌,73b5120b7df4e8b6c80200868795d998207ca3f5303155515c66367c7a4a1b0a


In [0]:
tweets_count_before_dedup = df_tweets_key.count()

df_tweets_dedup = df_tweets_key.dropDuplicates(["duplicate_key"])

tweets_count_after_dedup = df_tweets_dedup.count()
tweets_duplicates_removed = tweets_count_before_dedup - tweets_count_after_dedup

print("Tweets avant déduplication :", tweets_count_before_dedup)
print("Tweets après déduplication :", tweets_count_after_dedup)
print("Doublons supprimés :", tweets_duplicates_removed)


Tweets avant déduplication : 22524
Tweets après déduplication : 22514
Doublons supprimés : 10


In [0]:
def detect_language_safe(text):
    try:
        if text is None:
            return "unknown"
        
        text = text.strip()
        
        if len(text) < 5:
            return "unknown"
        
        return detect(text)
    
    except Exception:
        return "unknown"

detect_language_udf = udf(detect_language_safe, StringType())

In [0]:
df_tweets_lang = df_tweets_dedup.withColumn(
    "language",
    detect_language_udf(col("text_clean"))
)

df_tweets_lang = df_tweets_lang.withColumn(
    "language_group",
    when(col("language").isin("en", "fr", "es", "ar"), col("language"))
    .when(col("language") == "unknown", lit("unknown"))
    .otherwise(lit("other"))
)

display(df_tweets_lang.select("text_clean", "language", "language_group").limit(20))

text_clean,language,language_group
what a happy night #chz $pfl #socios #worldcup2022,en,en
"with your looks, you couldn't even get with a football player in the 4th division #qatarworldcup2022 #soccer #football #worldcup2022",en,en
yes we can 🇨🇦 #worldcup2022,en,en
fifa world cup day one- qatar 0 - ecuador 2 #worldcup2022 #fifaworldcup2022 #qatar2022 #ecuadorvsqatar #cr7𓃵,en,en
italy will win the world cup #worldcup2022 #fifaworldcup2022,en,en
check out christian pulisic - 2016-17 panini donruss debut gold parallel no. 224 - hga 9.0 #ebay via #worldcup #worldcup2022 #worldcupqatar2022,en,en
anti-regime iranian player watch: iran's hajsafi: conditions at home 'are not right' #worldcup2022 #iranprotests,en,en
saudi man refuses to be interviewed by an apartheid israeli tv channel during #worldcup2022 in qatar. ❤️🇵🇸,en,en
should of boycotted #worldcup2022 if you were that bothered instead of pointless kneeling & virtue signalling #englandfootball,en,en
proud of your jungkook! you killed it 🔥💜👏 #worldcup2022 #btsjungkook,af,other


In [0]:
df_tweets_quality = df_tweets_lang.withColumn(
    "quality_reason",
    when(col("text_clean").isNull(), "empty_text")
    .when(col("text_length") < 5, "too_short")
    .when(col("language_group") == "unknown", "unknown_language")
    .when(col("created_at_raw").isNull(), "missing_date")
    .otherwise("ok")
)

df_tweets_quality = df_tweets_quality.withColumn(
    "is_valid",
    when(col("quality_reason") == "ok", True).otherwise(False)
)

df_tweets_quality = df_tweets_quality.withColumn(
    "processing_ts",
    current_timestamp()
)

display(
    df_tweets_quality.select(
        "post_id", "text_clean", "language_group", "is_valid", "quality_reason"
    ).limit(20)
)

post_id,text_clean,language_group,is_valid,quality_reason
11,what a happy night #chz $pfl #socios #worldcup2022,en,true,ok
38,"with your looks, you couldn't even get with a football player in the 4th division #qatarworldcup2022 #soccer #football #worldcup2022",en,true,ok
42,yes we can 🇨🇦 #worldcup2022,en,true,ok
60,fifa world cup day one- qatar 0 - ecuador 2 #worldcup2022 #fifaworldcup2022 #qatar2022 #ecuadorvsqatar #cr7𓃵,en,true,ok
133,italy will win the world cup #worldcup2022 #fifaworldcup2022,en,true,ok
137,check out christian pulisic - 2016-17 panini donruss debut gold parallel no. 224 - hga 9.0 #ebay via #worldcup #worldcup2022 #worldcupqatar2022,en,true,ok
150,anti-regime iranian player watch: iran's hajsafi: conditions at home 'are not right' #worldcup2022 #iranprotests,en,true,ok
157,saudi man refuses to be interviewed by an apartheid israeli tv channel during #worldcup2022 in qatar. ❤️🇵🇸,en,true,ok
201,should of boycotted #worldcup2022 if you were that bothered instead of pointless kneeling & virtue signalling #englandfootball,en,true,ok
202,proud of your jungkook! you killed it 🔥💜👏 #worldcup2022 #btsjungkook,en,true,ok


In [0]:
df_silver_tweets = df_tweets_quality.select(
    "post_id",
    "source",
    "created_at_raw",
    "created_at",
    "match_name",
    "text_raw",
    "text_clean",
    "text_length",
    "has_url",
    "has_hashtag",
    "language",
    "language_group",
    "duplicate_key",
    "is_valid",
    "quality_reason",
    "bronze_source_file",
    "ingestion_ts",
    "processing_ts"
)

display(df_silver_tweets.limit(20))

post_id,source,created_at_raw,created_at,match_name,text_raw,text_clean,text_length,has_url,has_hashtag,language,language_group,duplicate_key,is_valid,quality_reason,bronze_source_file,ingestion_ts,processing_ts
11,Twitter for iPhone,2022-11-20 23:54:44,2022-11-20T23:54:44.000Z,world_cup_2022,What a happy night @socios @Socios_Turkiye #CHZ $PFL #socios #WorldCup2022 https://t.co/1sDsWvrztU,what a happy night #chz $pfl #socios #worldcup2022,50,true,true,en,en,3f0dfbd705edfd996cc2e5e7205dbbcc5f1541e7eb21b3221fce8b7616b31b60,true,ok,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:58:05.225Z,2026-07-02T11:58:05.225Z
38,Twitter Web App,2022-11-20 23:46:30,2022-11-20T23:46:30.000Z,world_cup_2022,"With your looks, you couldn't even get with a football player in the 4th division #QatarWorldCup2022 #Soccer #Football #WorldCup2022","with your looks, you couldn't even get with a football player in the 4th division #qatarworldcup2022 #soccer #football #worldcup2022",132,false,true,en,en,50a5ed494b824c3ef225b4402f1cabfb9a131f5697dfe8d1fd7f5e89f70c13ad,true,ok,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:58:05.225Z,2026-07-02T11:58:05.225Z
42,Twitter for Android,2022-11-20 23:45:15,2022-11-20T23:45:15.000Z,world_cup_2022,Yes we can 🇨🇦 #WorldCup2022,yes we can 🇨🇦 #worldcup2022,27,false,true,en,en,2d2fdb2050faab3c5c957233ea8fd7e5a1d4f7eb9a5b95dfcbbd093c962e3e79,true,ok,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:58:05.225Z,2026-07-02T11:58:05.225Z
60,Twitter Web App,2022-11-20 23:39:03,2022-11-20T23:39:03.000Z,world_cup_2022,https://t.co/Hl7BxTYlh3 Fifa World cup day one- Qatar 0 - Ecuador 2 #WorldCup2022 #FIFAWorldCup2022 #Qatar2022 #EcuadorvsQatar #CR7𓃵,fifa world cup day one- qatar 0 - ecuador 2 #worldcup2022 #fifaworldcup2022 #qatar2022 #ecuadorvsqatar #cr7𓃵,108,true,true,en,en,7ec381c950429f498f54c82ec6b0e39676f964fd76e190e6e737017f20efbee2,true,ok,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:58:05.225Z,2026-07-02T11:58:05.225Z
133,Twitter Web App,2022-11-20 23:18:21,2022-11-20T23:18:21.000Z,world_cup_2022,Italy will win the world cup #WorldCup2022 #FIFAWorldCup2022 https://t.co/RZ0uKWjGrP,italy will win the world cup #worldcup2022 #fifaworldcup2022,60,true,true,en,en,9df991b84cb56f81556ddd1af30867325f83e74553e04053e17c9ab1c33ef8e4,true,ok,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:58:05.225Z,2026-07-02T11:58:05.225Z
137,Twitter for Android,2022-11-20 23:18:04,2022-11-20T23:18:04.000Z,world_cup_2022,Check out Christian Pulisic - 2016-17 Panini Donruss Debut Gold Parallel No. 224 - HGA 9.0 https://t.co/BJWigLTymZ #eBay via @eBay #WorldCup #WorldCup2022 #WorldcupQatar2022,check out christian pulisic - 2016-17 panini donruss debut gold parallel no. 224 - hga 9.0 #ebay via #worldcup #worldcup2022 #worldcupqatar2022,143,true,true,en,en,d8307c70724b7c6e9b8bb3a374cd6f95a7feb2222cda85b31e15497afba1e02a,true,ok,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:58:05.225Z,2026-07-02T11:58:05.225Z
150,Twitter Web App,2022-11-20 23:16:01,2022-11-20T23:16:01.000Z,world_cup_2022,Anti-regime Iranian player watch: Iran's Hajsafi: Conditions at home 'are not right' #worldcup2022 #iranprotests https://t.co/nqW5oqjGpp,anti-regime iranian player watch: iran's hajsafi: conditions at home 'are not right' #worldcup2022 #iranprotests,112,true,true,en,en,1449f290a412fd69a38c64458a139002b1d8ccb007bc3190e59769f4f1750bfd,true,ok,file:/Workspace/Users/bentissesalma18@gmail.com/TP (1)/fifa_world_cup_2022_tweets.csv,2026-07-02T11:58:05.225Z,2026-07-02T11:58:05.225Z
157,Twitter for Android,2022-11-20 23:15:10,2022-11-20T23:15:10.000Z,world_cup_2022,Saudi man refuses to be interviewed by an apartheid Israeli TV channel during #WorldCup2022 in Qatar. ❤

In [0]:
event_match_col = find_column(df_events_raw, [
    "match_name", "match", "game", "fixture"
])

event_time_col = find_column(df_events_raw, [
    "event_time", "time", "minute", "match_minute", "Minute"
])

event_type_col = find_column(df_events_raw, [
    "event_type", "type", "event", "Event"
])

event_team_col = find_column(df_events_raw, [
    "team", "Team", "country", "nation"
])

event_player_col = find_column(df_events_raw, [
    "player", "Player", "player_name", "name"
])

event_description_col = find_column(df_events_raw, [
    "description", "Description", "details", "comment"
])

event_date_col = find_column(df_events_raw, [
    "created_at", "date", "timestamp", "Datetime", "event_date"
])

print("Colonnes détectées pour événements :")
print("event_match_col       =", event_match_col)
print("event_time_col        =", event_time_col)
print("event_type_col        =", event_type_col)
print("event_team_col        =", event_team_col)
print("event_player_col      =", event_player_col)
print("event_description_col =", event_description_col)
print("event_date_col        =", event_date_col)

Colonnes détectées pour événements :
event_match_col       = None
event_time_col        = None
event_type_col        = None
event_team_col        = None
event_player_col      = None
event_description_col = None
event_date_col        = None


In [0]:
df_silver_tweets.write.format("delta").mode("overwrite").saveAsTable("silver_posts_cleaned")
 

In [0]:
spark.table("silver_posts_cleaned").count()

22514

In [0]:
from pyspark.sql.functions import col
 
df_silver_for_nlp = (
    spark.table("silver_posts_cleaned")
    .filter(col("is_valid") == True)
    .select(
        "post_id",
        "source",
        "created_at",
        "match_name",
        "text_clean",
        "language_group",
        "is_valid"
    )
)
 
df_silver_for_nlp.write.format("delta").mode("overwrite").saveAsTable("silver_posts_cleaned_for_nlp")

In [0]:
df_check_nlp = spark.table("silver_posts_cleaned_for_nlp")
 
display(df_check_nlp.limit(20))

post_id,source,created_at,match_name,text_clean,language_group,is_valid
11,Twitter for iPhone,2022-11-20T23:54:44.000Z,world_cup_2022,what a happy night #chz $pfl #socios #worldcup2022,en,true
38,Twitter Web App,2022-11-20T23:46:30.000Z,world_cup_2022,"with your looks, you couldn't even get with a football player in the 4th division #qatarworldcup2022 #soccer #football #worldcup2022",en,true
42,Twitter for Android,2022-11-20T23:45:15.000Z,world_cup_2022,yes we can 🇨🇦 #worldcup2022,en,true
60,Twitter Web App,2022-11-20T23:39:03.000Z,world_cup_2022,fifa world cup day one- qatar 0 - ecuador 2 #worldcup2022 #fifaworldcup2022 #qatar2022 #ecuadorvsqatar #cr7𓃵,en,true
133,Twitter Web App,2022-11-20T23:18:21.000Z,world_cup_2022,italy will win the world cup #worldcup2022 #fifaworldcup2022,en,true
137,Twitter for Android,2022-11-20T23:18:04.000Z,world_cup_2022,check out christian pulisic - 2016-17 panini donruss debut gold parallel no. 224 - hga 9.0 #ebay via #worldcup #worldcup2022 #worldcupqatar2022,en,true
150,Twitter Web App,2022-11-20T23:16:01.000Z,world_cup_2022,anti-regime iranian player watch: iran's hajsafi: conditions at home 'are not right' #worldcup2022 #iranprotests,en,true
157,Twitter for Android,2022-11-20T23:15:10.000Z,world_cup_2022,saudi man refuses to be interviewed by an apartheid israeli tv channel during #worldcup2022 in qatar. ❤️🇵🇸,en,true
201,Twitter for iPhone,2022-11-20T23:07:49.000Z,world_cup_2022,should of boycotted #worldcup2022 if you were that bothered instead of pointless kneeling & virtue signalling #englandfootball,en,true
202,Twitter for Android,2022-11-20T23:07:46.000Z,world_cup_2022,proud of your jungkook! you killed it 🔥💜👏 #worldcup2022 #btsjungkook,en,true


In [0]:
from pyspark.sql.functions import col
df_silver_for_nlp = (
    spark.table("silver_posts_cleaned")
    .filter(col("is_valid") == True)
    .select(
        "post_id",
        "source",
        "created_at",
        "match_name",
        "text_clean",
        "language_group",
        "is_valid"
    )
)
df_silver_for_nlp.write.format("delta").mode("overwrite").saveAsTable("silver_posts_cleaned_for_nlp")

In [0]:
df_check = spark.table("silver_posts_cleaned_for_nlp")
 
display(df_check.limit(20))
print("Nombre de lignes :", df_check.count())
df_check.printSchema()

post_id,source,created_at,match_name,text_clean,language_group,is_valid
11,Twitter for iPhone,2022-11-20T23:54:44.000Z,world_cup_2022,what a happy night #chz $pfl #socios #worldcup2022,en,true
38,Twitter Web App,2022-11-20T23:46:30.000Z,world_cup_2022,"with your looks, you couldn't even get with a football player in the 4th division #qatarworldcup2022 #soccer #football #worldcup2022",en,true
42,Twitter for Android,2022-11-20T23:45:15.000Z,world_cup_2022,yes we can 🇨🇦 #worldcup2022,en,true
60,Twitter Web App,2022-11-20T23:39:03.000Z,world_cup_2022,fifa world cup day one- qatar 0 - ecuador 2 #worldcup2022 #fifaworldcup2022 #qatar2022 #ecuadorvsqatar #cr7𓃵,en,true
133,Twitter Web App,2022-11-20T23:18:21.000Z,world_cup_2022,italy will win the world cup #worldcup2022 #fifaworldcup2022,en,true
137,Twitter for Android,2022-11-20T23:18:04.000Z,world_cup_2022,check out christian pulisic - 2016-17 panini donruss debut gold parallel no. 224 - hga 9.0 #ebay via #worldcup #worldcup2022 #worldcupqatar2022,en,true
150,Twitter Web App,2022-11-20T23:16:01.000Z,world_cup_2022,anti-regime iranian player watch: iran's hajsafi: conditions at home 'are not right' #worldcup2022 #iranprotests,en,true
157,Twitter for Android,2022-11-20T23:15:10.000Z,world_cup_2022,saudi man refuses to be interviewed by an apartheid israeli tv channel during #worldcup2022 in qatar. ❤️🇵🇸,en,true
201,Twitter for iPhone,2022-11-20T23:07:49.000Z,world_cup_2022,should of boycotted #worldcup2022 if you were that bothered instead of pointless kneeling & virtue signalling #englandfootball,en,true
202,Twitter for Android,2022-11-20T23:07:46.000Z,world_cup_2022,proud of your jungkook! you killed it 🔥💜👏 #worldcup2022 #btsjungkook,en,true


Nombre de lignes : 22510
root
 |-- post_id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- match_name: string (nullable = true)
 |-- text_clean: string (nullable = true)
 |-- language_group: string (nullable = true)
 |-- is_valid: boolean (nullable = true)

